## Motivating Examples: Entity Recognition and Hallucination

Same query structure ("What sport does X play?"), differing only in entity recognition.

Case 1 (Simone Biles): entity recognized → correct output.

Case 2 (Katharina Ravichandran): entity not recognized → hallucination.

Motivates the Fake Template design. See Chapter II, Section II.A.
Model: GPT-4o | Temperature: 0.7 | March 2026

In [3]:
import os
import json
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

API_URL = "https://go.apis.huit.harvard.edu/ais-openai-direct-limited-schools/v1/chat/completions"
MODEL = "gpt-4o"
API_KEY = os.environ["HUIT_API_KEY"]

HEADERS = {
    "Content-Type": "application/json",
    "api-key": API_KEY,
}

PROMPTS = [
    "What sport does Simone Biles play?",
    "What sport does Katharina Ravichandran play?"
]

class TransientHTTPError(Exception):
    pass

def _raise_for_status(response):
    if response.status_code in (429, 500, 502, 503, 504):
        raise TransientHTTPError(response.text)
    if response.status_code >= 400:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text}")

@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def ask(prompt, temperature=0.7, max_tokens=300):
    response = requests.post(API_URL, headers=HEADERS, timeout=60, data=json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False,
        "n": 1,
    }))
    _raise_for_status(response)
    return response.json()["choices"][0]["message"]["content"]

for prompt in PROMPTS:
    print(f"PROMPT: {prompt}")
    print(f"OUTPUT: {ask(prompt)}")
    print()

PROMPT: What sport does Simone Biles play?
OUTPUT: Simone Biles is a gymnast. She is widely regarded as one of the greatest gymnasts of all time.

PROMPT: What sport does Katharina Ravichandran play?
OUTPUT: Katharina Ravichandran is a football (soccer) player.



PROMPT: What sport does Simone Biles play?

OUTPUT: Simone Biles is a gymnast.

PROMPT: What sport does Katharina Ravichandran play?

OUTPUT: Katharina Ravichandran is a cricketer.